# Hunyuan3D 2.1 — Batch Shape-Only Generation (NVIDIA T4)

- **Target Hardware**: NVIDIA T4 GPU (16 GB VRAM, shape-only fits in ~10 GB).
- **Inputs**: All PNG/JPG/JPEG files in `/kaggle/input/avalon-concepts/`.
- **Outputs**: Raw untextured `.glb` meshes in `/kaggle/working/raw/`.
- **Session Resilience**: Skips existing non-empty `.glb` files on restart.
- **Metrics**: Reports peak VRAM and wall-clock per asset.
- **Texture Pipeline**: Disabled (texturing performed downstream in Blender).

In [ ]:
# Cell 1: Repository clone and pinned dependencies for NVIDIA T4
import os
import sys

# Clone Hunyuan3D-2.1 repository if not already present
if not os.path.exists("Hunyuan3D-2.1"):
    !git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git

# Add repository to sys.path so hy3dshape can be imported directly
repo_path = os.path.abspath("Hunyuan3D-2.1")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Exact pinned dependencies (avoids bpy / custom texture rasterizer compilation)
!pip install -q --no-cache-dir \
    torch==2.5.1 \
    torchvision==0.20.1 \
    torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

!pip install -q --no-cache-dir \
    diffusers==0.31.0 \
    transformers==4.44.2 \
    accelerate==0.34.2 \
    trimesh==4.4.9 \
    einops==0.8.0 \
    safetensors==0.4.5 \
    rembg==2.0.65 \
    onnxruntime-gpu==1.19.2 \
    omegaconf==2.3.0 \
    pygltflib==1.16.2 \
    scipy==1.14.1 \
    numpy==1.26.4 \
    huggingface-hub==0.25.2 \
    ninja==1.11.1.1

In [ ]:
# Cell 2: GPU Check and Pipeline Initialization (Shape-Only)
import os
import sys
import gc
import time
from pathlib import Path
from PIL import Image
import torch

# Verify GPU device
assert torch.cuda.is_available(), "CUDA GPU is required! Select GPU T4 x2 or T4 x1 in Kaggle notebook settings."
device_name = torch.cuda.get_device_name(0)
device_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Active GPU: {device_name} ({device_mem:.2f} GB total VRAM)")

# Import Hunyuan3D pipeline and background remover
try:
    from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
    from hy3dshape.rembg import BackgroundRemover
except ImportError:
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    from hy3dgen.rembg import BackgroundRemover

INPUT_DIR = Path("/kaggle/input/avalon-concepts")
OUTPUT_DIR = Path("/kaggle/working/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "tencent/Hunyuan3D-2.1"
SUBFOLDER = "hunyuan3d-dit-v2-1"

print(f"Loading shape model '{MODEL_ID}' (FP16)... ")
try:
    pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
        MODEL_ID,
        subfolder=SUBFOLDER,
        torch_dtype=torch.float16
    )
except Exception:
    pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16
    )

pipeline = pipeline.to("cuda")
rembg = BackgroundRemover()
print("Hunyuan3D 2.1 Shape-Only Pipeline ready on CUDA.")

In [ ]:
# Cell 3: Batch Execution over /kaggle/input/avalon-concepts/
# - Supports PNG, JPG, JPEG
# - Skips assets whose GLB already exists (survives restart)
# - Prints peak VRAM and wall-clock per asset
# - No interactive widgets

valid_exts = {".png", ".jpg", ".jpeg"}
image_files = sorted([p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in valid_exts])

# Deduplicate paths
seen = set()
unique_images = []
for p in image_files:
    if p.resolve() not in seen:
        seen.add(p.resolve())
        unique_images.append(p)

print(f"Found {len(unique_images)} concept image(s) in {INPUT_DIR}\n")

processed = 0
skipped = 0
failed = 0

for idx, img_file in enumerate(unique_images, 1):
    asset_name = img_file.stem
    target_glb = OUTPUT_DIR / f"{asset_name}.glb"

    # Restart resilience: skip completed non-empty GLBs
    if target_glb.exists() and target_glb.stat().st_size > 0:
        print(f"[{idx}/{len(unique_images)}] SKIP: {target_glb.name} already exists ({target_glb.stat().st_size / 1024:.1f} KB)")
        skipped += 1
        continue

    print(f"[{idx}/{len(unique_images)}] START: {asset_name} ({img_file.name})")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t_start = time.perf_counter()

    try:
        # Load image & remove background if solid/opaque
        image = Image.open(img_file).convert("RGBA")
        extrema = image.getextrema()
        if len(extrema) < 4 or extrema[3][0] == 255:
            image = rembg(image)

        # Run shape pipeline in inference mode
        with torch.inference_mode():
            mesh = pipeline(
                image=image,
                num_inference_steps=30,
                octree_resolution=256,
                num_chunks=20000,
                generator=torch.manual_seed(42),
                output_type="trimesh"
            )[0]

        # Export untextured GLB
        mesh.export(str(target_glb), file_type="glb")

        torch.cuda.synchronize()
        wall_clock = time.perf_counter() - t_start
        peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
        peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        glb_size_kb = target_glb.stat().st_size / 1024

        print(
            f"[{idx}/{len(unique_images)}] DONE: {asset_name} -> {target_glb.name} ({glb_size_kb:.1f} KB) | "
            f"Wall-clock: {wall_clock:.2f}s | Peak VRAM: {peak_vram_gb:.2f} GB ({peak_vram_mb:.1f} MB)"
        )
        processed += 1

    except Exception as e:
        torch.cuda.synchronize()
        wall_clock = time.perf_counter() - t_start
        print(f"[{idx}/{len(unique_images)}] ERROR on {asset_name}: {e} (after {wall_clock:.2f}s)")
        failed += 1
        if target_glb.exists():
            try:
                target_glb.unlink()
            except OSError:
                pass
    finally:
        gc.collect()
        torch.cuda.empty_cache()

print("-" * 70)
print(f"Batch Finished: Processed={processed}, Skipped={skipped}, Failed={failed}, Total={len(unique_images)}")

In [ ]:
# Cell 4: Compress /kaggle/working/raw for Download
import shutil
from pathlib import Path

raw_dir = Path("/kaggle/working/raw")
zip_base = Path("/kaggle/working/avalon_raw_glb")

glb_files = sorted(list(raw_dir.glob("*.glb")))
print(f"Total GLB files in {raw_dir}: {len(glb_files)}\n")
for glb in glb_files:
    print(f"  - {glb.name} ({glb.stat().st_size / 1024:.1f} KB)")

if glb_files:
    archive_path = shutil.make_archive(
        base_name=str(zip_base),
        format="zip",
        root_dir=str(raw_dir)
    )
    size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f"\nZip archive created: {archive_path} ({size_mb:.2f} MB)")
    print("Download avalon_raw_glb.zip from the Kaggle output file tree.")
else:
    print("\nWarning: No GLB files found in raw directory to zip.")